# Price statistical baselines (Darts)



In [ ]:
!pip install -r requirements/darts.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"price"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

from darts import TimeSeries, concatenate
from darts.models import NaiveSeasonal, AutoARIMA, AutoTheta, AutoETS
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from preprocessing import load_data, FUTURE_COVARIATES
from gefcom import (
    load_gefcom_price,
    evaluate_gefcom_price,
    PRICE_COVARIATES,
    GEFCOM_PRICE_ALL_DATES,
)
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
NUM_SAMPLES=100

OUTPUT_DIR=ROOT/"outputs"/"predictions"
MODEL_DIR=ROOT/"models"/"darts"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
MODEL_DIR.mkdir(parents=True,exist_ok=True)

def build_stat(name):
    if name=="auto_arima":
        return AutoARIMA(
            seasonal=True,
            season_length=24,
            stepwise=True,
            trace=True,
            d=1,D=1,
            max_p=1,max_q=1,max_P=1,max_Q=1,
        )
    if name=="auto_theta":
        return AutoTheta(season_length=24)
    if name=="auto_ets":
        return AutoETS(season_length=24)
    raise ValueError(name)

STAT_CLASSES={"auto_arima":AutoARIMA,"auto_theta":AutoTheta,"auto_ets":AutoETS}

def probabilistic_frame(series,forecast):
    gt=series.slice_intersect(forecast)
    fc=forecast.slice_intersect(gt)
    out=pd.DataFrame({
        "item_id":1,
        "timestamp":gt.time_index,
        "target":gt.values().flatten(),
    })
    for q in OUTPUT_QUANTILES:
        out[str(q)]=fc.quantile(q).values().flatten()
    out["prediction"]=out["0.5"]
    return out

def deterministic_frame(series,forecast):
    gt=series.slice_intersect(forecast)
    fc=forecast.slice_intersect(gt)
    return pd.DataFrame({
        "item_id":1,
        "timestamp":gt.time_index,
        "target":gt.values().flatten(),
        "prediction":fc.values().flatten(),
    })


## GEFCom2014-P


In [ ]:
GEF_LOAD_EXISTING=False
GEF_SAVE_MODELS=True
GEF_MODEL_DIR=MODEL_DIR/"gefcom"
GEF_MODEL_DIR.mkdir(parents=True,exist_ok=True)

gef_df=load_gefcom_price(ROOT/"data"/"gefcom_price_data.csv")
gef_series=TimeSeries.from_dataframe(
    gef_df,time_col="timestamp",value_cols="target",freq="h"
).astype(np.float32)
gef_future_covariates=TimeSeries.from_dataframe(
    gef_df,time_col="timestamp",value_cols=PRICE_COVARIATES,freq="h"
).astype(np.float32)

gef_cutoff=pd.Timestamp("2013-06-15 23:00:00")
gef_train,_=gef_series.split_after(gef_cutoff)
gef_future_train,_=gef_future_covariates.split_after(gef_cutoff)

print("GEF target:",gef_series.start_time(),"->",gef_series.end_time())
print("GEF future covariates:",gef_future_covariates.start_time(),"->",gef_future_covariates.end_time())
assert gef_future_covariates.end_time()>=gef_series.end_time()


In [ ]:
# Seasonal Naive: deterministic.
gef_naive=NaiveSeasonal(K=24)
gef_naive.fit(gef_train)
gef_bt=gef_naive.historical_forecasts(
    series=gef_series,
    start=gef_cutoff+pd.Timedelta(hours=1),
    forecast_horizon=24,
    stride=24,
    retrain=True,
    last_points_only=False,
    verbose=True,
)
gef_fc=concatenate(gef_bt)
gef_naive_result=deterministic_frame(gef_series,gef_fc)
gef_naive_result["task_date"]=pd.to_datetime(gef_naive_result["timestamp"]).dt.floor("D")
gef_naive_result=gef_naive_result[
    gef_naive_result["task_date"].isin(GEFCOM_PRICE_ALL_DATES)
].drop(columns="task_date").reset_index(drop=True)
gef_naive_result.to_csv(OUTPUT_DIR/"seasonal_naive_gefcom_price.csv",index=False)
print("GEFCom Seasonal Naive")
print("MAE:",mean_absolute_error(gef_naive_result.target,gef_naive_result.prediction))
print("RMSE:",root_mean_squared_error(gef_naive_result.target,gef_naive_result.prediction))
print("aQL: N/A (deterministic baseline)")


In [ ]:
for name in ["auto_arima","auto_theta","auto_ets"]:
    path=GEF_MODEL_DIR/f"{name}.darts"

    if GEF_LOAD_EXISTING and path.exists():
        model=STAT_CLASSES[name].load(str(path))
    else:
        model=build_stat(name)
        model.fit(
            series=gef_train,
            future_covariates=gef_future_train,
        )
        if GEF_SAVE_MODELS:
            model.save(str(path))

    bt=model.historical_forecasts(
        series=gef_series,
        future_covariates=gef_future_covariates,
        start=gef_cutoff+pd.Timedelta(hours=1),
        forecast_horizon=24,
        stride=24,
        num_samples=NUM_SAMPLES,
        retrain=False,
        last_points_only=False,
        verbose=True,
    )
    forecast=concatenate(bt)
    result=probabilistic_frame(gef_series,forecast)

    result["task_date"]=pd.to_datetime(result["timestamp"]).dt.floor("D")
    result=result[
        result["task_date"].isin(GEFCOM_PRICE_ALL_DATES)
    ].drop(columns="task_date").reset_index(drop=True)

    expected=24*len(GEFCOM_PRICE_ALL_DATES)
    if len(result)!=expected:
        raise ValueError(f"Expected {expected} GEFCom price rows, got {len(result)}.")

    result.to_csv(OUTPUT_DIR/f"{name}_gefcom_price.csv",index=False)
    summary=evaluate_gefcom_price(result,BASE_QUANTILES)
    display(summary[["task_date","raw_aql","normalized_aql"]])
    summary.to_csv(OUTPUT_DIR/f"{name}_gefcom_price_summary.csv",index=False)


## GridStatus2025


In [ ]:
LOAD_EXISTING=False
SAVE_MODELS=True
GRID_MODEL_DIR=MODEL_DIR/"gridstatus"
GRID_MODEL_DIR.mkdir(parents=True,exist_ok=True)

data=load_data(ROOT/"data")
series=TimeSeries.from_dataframe(
    data.full,time_col="timestamp",value_cols="target",freq="h"
).astype(np.float32)
future_covariates=TimeSeries.from_dataframe(
    data.full,time_col="timestamp",value_cols=FUTURE_COVARIATES,freq="h"
).astype(np.float32)

test_start=pd.Timestamp(data.test["timestamp"].min())
stat_train,_=series.split_after(test_start-series.freq)
future_train,_=future_covariates.split_after(test_start-series.freq)

print("Target:",series.start_time(),"->",series.end_time())
print("Future covariates:",future_covariates.start_time(),"->",future_covariates.end_time())
assert future_covariates.end_time()>=series.end_time()


In [ ]:
seasonal_naive=NaiveSeasonal(K=24)
seasonal_naive.fit(stat_train)
bt=seasonal_naive.historical_forecasts(
    series=series,
    start=test_start,
    forecast_horizon=24,
    stride=24,
    retrain=True,
    last_points_only=False,
    verbose=True,
)
forecast=concatenate(bt)
result=deterministic_frame(series,forecast)
for q in OUTPUT_QUANTILES:
    result[str(q)]=np.nan
result.to_csv(OUTPUT_DIR/"seasonal_naive.csv",index=False)
print("Seasonal Naive")
print(f"MAE:  {mean_absolute_error(result.target,result.prediction):.6f}")
print(f"RMSE: {root_mean_squared_error(result.target,result.prediction):.6f}")
print("aQL:  N/A (deterministic baseline)")


In [ ]:
for name in ["auto_arima","auto_theta","auto_ets"]:
    path=GRID_MODEL_DIR/f"{name}.darts"

    if LOAD_EXISTING and path.exists():
        model=STAT_CLASSES[name].load(str(path))
    else:
        model=build_stat(name)
        model.fit(
            series=stat_train,
            future_covariates=future_train,
        )
        if SAVE_MODELS:
            model.save(str(path))

    bt=model.historical_forecasts(
        series=series,
        future_covariates=future_covariates,
        start=test_start,
        forecast_horizon=24,
        stride=24,
        num_samples=NUM_SAMPLES,
        retrain=False,
        last_points_only=False,
        verbose=True,
    )
    forecast=concatenate(bt)
    result=probabilistic_frame(series,forecast)
    result.to_csv(OUTPUT_DIR/f"{name}.csv",index=False)
    print_metrics(result,BASE_QUANTILES,label=name)
